# 02 · Working with tabular data in pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oechenique/python-gis-basics/blob/main/notebooks/02_Working_With_Data.ipynb)

Before geometry comes the table. This notebook writes a small CSV of 12 Argentine cities (with two problems on purpose), reads it back, cleans it, adds columns and summarises it by region. No downloads.

## 1. A CSV with problems

In real life the file comes from somewhere else. Here we write it ourselves so the notebook runs anywhere, with one missing region and one latitude stored as text with a comma.

In [1]:
from pathlib import Path

import pandas as pd

RAW_CSV = """city,province,region,lat,lon
Buenos Aires,CABA,Pampeana,-34.6037,-58.3816
Córdoba,Córdoba,Pampeana,-31.4201,-64.1888
Rosario,Santa Fe,Pampeana,-32.9468,-60.6393
Mar del Plata,Buenos Aires,,-38.0055,-57.5426
Mendoza,Mendoza,Cuyo,-32.8895,-68.8458
San Miguel de Tucumán,Tucumán,NOA,-26.8083,-65.2176
Salta,Salta,NOA,-24.7821,-65.4232
Posadas,Misiones,NEA,-27.3671,-55.8961
Resistencia,Chaco,NEA,"-27,4606",-58.9839
Neuquén,Neuquén,Patagonia,-38.9516,-68.0591
San Carlos de Bariloche,Río Negro,Patagonia,-41.1335,-71.3103
Ushuaia,Tierra del Fuego,Patagonia,-54.8019,-68.3030
"""

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / "cities_raw.csv"
csv_path.write_text(RAW_CSV, encoding="utf-8")

df = pd.read_csv(csv_path)
df.head()

,city,province,region,lat,lon
0,Buenos Aires,CABA,Pampeana,-34.6037,-58.3816
1,Córdoba,Córdoba,Pampeana,-31.4201,-64.1888
2,Rosario,Santa Fe,Pampeana,-32.9468,-60.6393
3,Mar del Plata,Buenos Aires,NaN,-38.0055,-57.5426
4,Mendoza,Mendoza,Cuyo,-32.8895,-68.8458


## 2. Inspect before trusting

`dtypes` shows that `lat` was read as text (the comma in Resistencia), and `isna()` finds the missing region.

In [2]:
print(df.dtypes, "\n")
print("Missing values per column:")
print(df.isna().sum())

city            str
province        str
region          str
lat             str
lon         float64
dtype: object 

Missing values per column:
city        0
province    0
region      1
lat         0
lon         0
dtype: int64


## 3. Clean

Fix the decimal comma and convert to numbers, then fill the region from a province lookup instead of guessing. Each step assigns back to the column (no `inplace=True`, which pandas 3 no longer supports on a column).

In [3]:
df["lat"] = pd.to_numeric(df["lat"].astype(str).str.replace(",", ".", regex=False))

REGION_BY_PROVINCE = {"Buenos Aires": "Pampeana"}
df["region"] = df["region"].fillna(df["province"].map(REGION_BY_PROVINCE))

assert df["lat"].between(-90, 90).all() and df["lon"].between(-180, 180).all()
assert df.notna().all().all()
df.dtypes

city            str
province        str
region          str
lat         float64
lon         float64
dtype: object

## 4. New columns

Distance to Buenos Aires (haversine, from notebook 01) and a latitude band with `pd.cut`.

In [4]:
import numpy as np

BA_LAT, BA_LON = -34.6037, -58.3816


def haversine_km(lat1, lon1, lat2, lon2, r=6371.0):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


df["km_to_ba"] = haversine_km(BA_LAT, BA_LON, df["lat"], df["lon"]).round(0)
df["band"] = pd.cut(df["lat"], bins=[-90, -40, -30, -20], labels=["south of 40°S", "30°S–40°S", "20°S–30°S"])
df.sort_values("km_to_ba")

,city,province,region,lat,lon,km_to_ba,band
0,Buenos Aires,CABA,Pampeana,-34.6037,-58.3816,0.0,30°S–40°S
2,Rosario,Santa Fe,Pampeana,-32.9468,-60.6393,278.0,30°S–40°S
3,Mar del Plata,Buenos Aires,Pampeana,-38.0055,-57.5426,386.0,30°S–40°S
1,Córdoba,Córdoba,Pampeana,-31.4201,-64.1888,647.0,30°S–40°S
8,Resistencia,Chaco,NEA,-27.4606,-58.9839,796.0,20°S–30°S
7,Posadas,Misiones,NEA,-27.3671,-55.8961,839.0,20°S–30°S
4,Mendoza,Mendoza,Cuyo,-32.8895,-68.8458,986.0,30°S–40°S
9,Neuquén,Neuquén,Patagonia,-38.9516,-68.0591,987.0,30°S–40°S
5,San Miguel de Tucumán,Tucumán,NOA,-26.8083,-65.2176,1085.0,20°S–30°S
6,Salta,Salta,NOA,-24.7821,-65.4232,1286.0,20°S–30°S


## 5. Summarise by region

In [5]:
summary = (
    df.groupby("region")
    .agg(cities=("city", "count"), mean_km_to_ba=("km_to_ba", "mean"), southernmost_lat=("lat", "min"))
    .round(0)
    .sort_values("mean_km_to_ba")
)
summary

,cities,mean_km_to_ba,southernmost_lat
region,,,
Pampeana,4,328.0,-38.0
NEA,2,818.0,-27.0
Cuyo,1,986.0,-33.0
NOA,2,1186.0,-27.0
Patagonia,3,1568.0,-55.0


## 6. Save the clean table

The raw file stays untouched; the result goes to a new file.

In [6]:
clean_path = data_dir / "cities_clean.csv"
df.to_csv(clean_path, index=False)
print(f"{clean_path}: {len(df)} rows, {len(df.columns)} columns")

data\cities_clean.csv: 12 rows, 7 columns


## Takeaways

- Read, then check `dtypes` and `isna()`: one stray comma turns a numeric column into text without any error.
- Fill missing values from a rule you can explain (a lookup), not from the mean.
- Keep the raw file and write the clean one next to it.
- `lat`/`lon` are still just numbers here. Notebook [03](03_Intro_To_Geodata.ipynb) turns them into geometries.

## Exercises

1. Add a `population` column for 3 cities and leave the rest empty. How does `groupby(...).mean()` treat the gaps?
2. Use `pd.cut` on `km_to_ba` to build "near / medium / far" classes.
3. Save the clean table as Parquet (`df.to_parquet`) and compare the file size with the CSV.